<a href="https://colab.research.google.com/github/eruu-data29/Fake-Review-Detection/blob/main/Fake_Review_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit pyngrok
!pip install streamlit scikit-learn wordcloud plotly
!streamlit run app.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.8 MB/s eta 0:00:00
Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [3]:
# ===============================
# Fake Review Detection Dashboard
# ===============================

import streamlit as st
import pandas as pd
import numpy as np
import re
import string

# NLP Libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from wordcloud import WordCloud
import matplotlib.pyplot as plt

import plotly.express as px

# ===============================
# PAGE CONFIG
# ===============================
st.set_page_config(page_title="Fake Review Detection Dashboard", layout="wide")

st.title("🕵️ Fake Review Detection Dashboard")

# ===============================
# TEXT CLEANING FUNCTION
# ===============================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

# ===============================
# LOAD DATA
# ===============================
st.sidebar.header("Upload Dataset")

uploaded_file = st.sidebar.file_uploader("Upload CSV", type=["csv"])

if uploaded_file:
    df = pd.read_csv(uploaded_file)

    st.subheader("📊 Raw Data")
    st.dataframe(df.head(), use_container_width=True)

    # ===============================
    # DATA CLEANING
    # ===============================
    if "review" not in df.columns or "label" not in df.columns:
        st.error("Dataset must contain 'review' and 'label' columns")
    else:
        df['clean_review'] = df['review'].apply(clean_text)

        # ===============================
        # TRAIN MODEL
        # ===============================
        vectorizer = TfidfVectorizer(max_features=5000)
        X = vectorizer.fit_transform(df['clean_review'])
        y = df['label']

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        model = LogisticRegression()
        model.fit(X_train, y_train)

        accuracy = model.score(X_test, y_test)

        st.success(f"Model Accuracy: {accuracy:.2f}")

        # ===============================
        # SIDEBAR INPUT
        # ===============================
        st.sidebar.subheader("🔍 Test a Review")

        user_input = st.sidebar.text_area("Enter Review")

        if st.sidebar.button("Predict"):
            cleaned = clean_text(user_input)
            vec = vectorizer.transform([cleaned])
            pred = model.predict(vec)[0]

            if pred == 1:
                st.sidebar.error("🚨 Fake Review Detected")
            else:
                st.sidebar.success("✅ Genuine Review")

        # ===============================
        # TABS
        # ===============================
        tab1, tab2, tab3, tab4 = st.tabs(
            ["📊 Overview", "📈 Visualizations", "☁️ WordCloud", "🔎 Search"]
        )

        # ===============================
        # TAB 1 - OVERVIEW
        # ===============================
        with tab1:
            st.subheader("Dataset Overview")

            col1, col2 = st.columns(2)

            with col1:
                st.metric("Total Reviews", len(df))

            with col2:
                fake_count = df['label'].sum()
                st.metric("Fake Reviews", fake_count)

        # ===============================
        # TAB 2 - VISUALIZATION
        # ===============================
        with tab2:
            st.subheader("Review Distribution")

            fig = px.pie(df, names='label', title="Fake vs Real Reviews")
            st.plotly_chart(fig, use_container_width=True)

            st.subheader("Review Length Analysis")
            df['length'] = df['review'].apply(len)

            fig2 = px.histogram(df, x="length", nbins=50)
            st.plotly_chart(fig2, use_container_width=True)

        # ===============================
        # TAB 3 - WORDCLOUD
        # ===============================
        with tab3:
            st.subheader("WordCloud of Reviews")

            text = " ".join(df['clean_review'])

            wc = WordCloud(width=800, height=400).generate(text)

            fig, ax = plt.subplots()
            ax.imshow(wc)
            ax.axis("off")

            st.pyplot(fig)

        # ===============================
        # TAB 4 - SEARCH
        # ===============================
        with tab4:
            st.subheader("Search Reviews")

            query = st.text_input("Enter keyword")

            if query:
                results = df[df['review'].str.contains(query, case=False, na=False)]

                if not results.empty:
                    st.success(f"Found {len(results)} reviews")
                    st.dataframe(results.head(), use_container_width=True)
                else:
                    st.warning("No results found")

else:
    st.info("Please upload a dataset to start")

2026-04-01 04:19:58.892 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 04:19:58.895 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 04:19:59.558 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-01 04:19:59.571 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 04:19:59.573 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 04:19:59.576 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 04:19:59.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn